In [1]:
#----------------------------------
#Work arround code for printing latex in VSCode. If you are running directly in JupiterLab, comment the following lines and run %display latex
from latex_disp_VSCode import slatex
#----------------------------------

def Xvar(n : int):
    X=[]
    for i in range(n):
        X.append('x'+str(i+1))
    return X
def Tvar(n : int):
    T=[]
    for i in range(n):
        T.append('T'+str(i+1))
    return T

#Function to define a supercommutative algebra with n even and l odd generators, whose name is given by list of strings in *lambda_lenghts, *mu_invariants.
def superalgebra(n : int,l : int, lambda_lenghts: list[[str]] = None, mu_invariants: list[[str]] = None) : 
    if lambda_lenghts is None : 
        P = PolynomialRing(QQ, n, Xvar(n)) ## n is for lamba lenghts
        P.inject_variables(verbose=False)
        P = FractionField(P)
    else : 
        if n != len(lambda_lenghts) : 
            print('The number of generators of the even part should match the length your names vector')
            return
        else: 
            P = PolynomialRing(QQ, n, lambda_lenghts)
            P.inject_variables(verbose=False)
            P = FractionField(P)
    #We will work with the exterior algebra over the odd variables
    if mu_invariants is None : 
        F = ExteriorAlgebra(P, names = Tvar(l)) ## l is for the mu invariants 
        F.inject_variables(verbose=False)
    else : 
        if l != len(mu_invariants) : 
            print('The number of generators of the odd part should match the length your names vector')
            return
        else: 
            F = ExteriorAlgebra(P, names = mu_invariants)
            F.inject_variables(verbose=False)
    return F

In [2]:
# ---------------------------------------------------------------------------
# Entries of a superfrieze of period P from its quiddity data, plus functions that test supergrowth analogues:
# - testes the difference  m_{i,j} - m_{i+1,j-1}  (j - i = P - 1, that is, the difference between entries two rows apart in rows P, P-2; its value is the principal growth coefficient s_1, in the classical case.
# - the recursion relation bewteen growths: 
#
#                           m_{i,j+2P} = s_1 m_{i,j+P} - m_{i,j}.
#
# Superfrieze conventions follow Mourier-Genoud, Ovsienko and Tabachnikov:
#   * even entries f_{i,j}, odd entries phi_{i,j}, half-integer odd entries phi_{i-1/2,j+1/2};
#     i indexes SE-diagonals, j indexes NE-diagonals.
#   * a generic superfrieze is determined by its first two non-trivial rows
#     a_i = f_{i,i} (even quiddity, usually dneoted L for lambda lenghts) and beta_i = phi_{i,i} (odd quiddity, usually E or T) via the
#     recurrence 
#                   (f_{i,i-3}, f_{i,i-2}, phi_{i,i-2}) = (-1,0,0),
#     then left-multiply by A_k = [[0,1,0],[-1,a_k,-beta_k],[0,beta_k,1]] for k = i-1, \cdots, j.
#   * row r (r = 1 is the even quiddity row f_{i,i}) = the cells with j - i = r - 1.
#
# If F is a superfrieze of an A(m,n) annulus, there are two types of infinite super friezes for the inner boundary by Theorem 3.2:
#   * n-periodic   : every row has period n.
#   * 2n-skew-periodic  : the even rows have period n, the odd rows period 2n since it is antiperiodic.
# Supply the quiddity data in one of the following forms:
#   * n-periodic  : L, E both length n;  odd_antiperiodic=False  (default).
#   * 2n-periodic : L, E both length n with odd_antiperiodic=True (you can also give the full length 2n quiddity rows, with Ep = E + (-E), and use the previous version)
# The period is  P = superfrieze_period(n, odd_antiperiodic)  ( = n or 2n ), and
# every growth-coefficient function below uses P (not n) as the "translation" to compute the growth coefficients. 
# The function even_growth compares f_{i,i+P-1} with f_{i+1,i+P-2}, test_growth sweeps i over a lenght P fundamental domain, check_recursion uses P and 2P.  detect_period(L,E,F,...) reports the period it actually finds and compares with the period you gave.
#
# Usage: F = superalgebra(n_even, n_odd); pass L = [a_0,...], E = [beta_0,...] as lists.
# ---------------------------------------------------------------------------



#First, define a function for super entries
def super_entry(L, E, i, j, n=None, odd_antiperiodic=False):
    #Return [f_{i,j-1}, f_{i,j}, phi_{i,j}] for the superfrieze with even quiddity L
    #and odd quiddity E, (default len(L)) is the quiddity length

    #odd_antiperiodic=False (default): odd quiddity genuinely n-periodic, beta_{k+n}=beta_k
    #odd_antiperiodic=True: beta_{k+n} = -beta_k -- 2n-periodic. The even quiddity is n-periodic in both.
    if len(L) != len(E):
        raise ValueError('even and odd quiddity sequences must have equal length')
    if n is None:
        n = len(L)
    v = [-1, 0, 0]
    for k in range(i - 1, j + 1):
        a = L[k % n]
        b = E[k % n] * ((-1) ** (k // n) if odd_antiperiodic else 1)
        A = [[0, 1, 0], [-1, a, -b], [0, b, 1]]
        v = [sum(A[r][c] * v[c] for c in range(3)) for r in range(3)]
    return v


## This function computes the "halved-indexed" odd-entries of the superfrieze
def super_entry_half(L, E, i, j, n=None, odd_antiperiodic=False):
    #Return the half-integer odd entry phi_{i-1/2, j+1/2} = f_{i,j} phi_{i-1,j} - f_{i-1,j} phi_{i,j}   
    a = super_entry(L, E, i,     j, n, odd_antiperiodic)
    b = super_entry(L, E, i - 1, j, n, odd_antiperiodic)
    return a[1] * b[2] - b[1] * a[2]


def superfrieze_period(n, odd_antiperiodic=False):
    #Actual period P of the superfrieze: n if every row is n-periodic, or 2n if the odd rows are only 2n-periodic (odd_antiperiodic=True)."""
    return 2 * n if odd_antiperiodic else n

# ---------------------------------------------------------------------------
# Split a superalgebra element into its even and the odd part
# ---------------------------------------------------------------------------

def body(x, F):
    #This function coerces an elemnt x in its exterior algebra F. It then takes all the coefficients in the odd variables and makes them equal to 0. 
    x = F(x)
    for mon, coeff in x.monomial_coefficients().items(): # mon, coeff is the unpacking of the tuple (mon, coef) \in  x.monomial_coefficients().items()
        if len(mon) == 0:
            return coeff
    return F.base_ring()(0)


def odd_correction(x, F):
    #x - body(x): the part of x built from the odd generators
    return F(x) - F(body(x, F))


def _is_zero(x, F):
    return F(x).is_zero()

# ---------------------------------------------------------------------------
# candidate growth quantities: the superfrieze analogue of  m_{i,j} - m_{i+1,j-1}
# for j - i = P - 1 (rows P and P-2), whose classical value is s_1.
# P is the superfrieze period (n or 2n). For the odd rows we used the "straightforward" idea of growth. 
# ---------------------------------------------------------------------------

def even_growth(L, E, i, n=None, odd_antiperiodic=False):
    #f_{i, i+P-1} - f_{i+1, i+P-2},  P = superfrieze_period(n, odd_antiperiodic)
    if n is None:
        n = len(L)
    P = superfrieze_period(n, odd_antiperiodic)
    return (super_entry(L, E, i,     i + P - 1, n, odd_antiperiodic)[1]
          - super_entry(L, E, i + 1, i + P - 2, n, odd_antiperiodic)[1])

##  "straightforward" odd growth
def odd_growth(L, E, i, n=None, odd_antiperiodic=False):
    #phi_{i, i+P-1} - phi_{i+1, i+P-2}
    if n is None:
        n = len(L)
    P = superfrieze_period(n, odd_antiperiodic)
    return (super_entry(L, E, i,     i + P - 1, n, odd_antiperiodic)[2]
          - super_entry(L, E, i + 1, i + P - 2, n, odd_antiperiodic)[2])

## "straightforward" halved-odd growth
def odd_growth_half(L, E, i, n=None, odd_antiperiodic=False):
    #phi_{i-1/2, i+P-3/2} - phi_{i+1/2, i+P-5/2}  (half-integer odd row)
    if n is None:
        n = len(L)
    P = superfrieze_period(n, odd_antiperiodic)
    return (super_entry_half(L, E, i,     i + P - 2, n, odd_antiperiodic)
          - super_entry_half(L, E, i + 1, i + P - 3, n, odd_antiperiodic))

def growth_coeffs(L, E, i, n=None, F=None, odd_antiperiodic=False):
    #Proposed growth coefficients at the SE-diagonal i. If the superalgebra F is given, also returns 'even_body', (the classical principal growth coefficient).
    if n is None:
        n = len(L)
    eg = even_growth(L, E, i, n, odd_antiperiodic)
    out = {
        'even':     even_growth(L, E, i, n, odd_antiperiodic),
        'odd':      odd_growth(L, E, i, n, odd_antiperiodic),
        'odd_half': odd_growth_half(L, E, i, n, odd_antiperiodic),
    }
    if F is not None:
        out['even_body'] = body(eg, F)
    return out

# ---------------------------------------------------------------------------
# Test functions, for bigger scale experiments
# ---------------------------------------------------------------------------

## Auxiliar function
def _growth_fn(kind):
    return {'even': even_growth, 'odd': odd_growth, 'odd_half': odd_growth_half}[kind]

# Here L is the quiddity sequence of lambda-lengths, E is the q.s. of mu-invariants, F is the super algebra, n is the period (optional), kind is the type of growth you want to check, reps is the number of times you do the horizontal shift,  odd_antiperiodic tells the code the type of the infinite super frieze, and kind is the type of growth you want to check (even, odd, half-odd)
def test_growth(L, E, F, n=None, kind='even', reps=2, odd_antiperiodic=False):
    """Theorem-2.2 analogue: Evaluate the chosen growth quantity
    for i = 0 .. reps*P-1 and report:
      P             -- the superfrieze period used (n or 2n)
      full_constant -- is the full superalgebra element associated to the growth coefficient independent of i?
      classic_constant -- This is exactly the classical growth coefficient theorem, expected super growth coeffients 
      body_value    -- the common body value (principal growth coefficient s_1)
      values        -- the list of the reps*P evaluated elements
    """
    if n is None:
        n = len(L)
    P = superfrieze_period(n, odd_antiperiodic)
    fn = _growth_fn(kind)
    vals = [fn(L, E, i, n, odd_antiperiodic) for i in range(reps * P)] # Vector of differences
    # vals = [fn(L, E, i, n, odd_antiperiodic) for i in range(reps * P if odd_antiperiodic else P)]
    full_constant = all(_is_zero(v - vals[0], F) for v in vals) # Checks whether the difference is constant 
    bodies = [body(v, F) for v in vals] # takes vals and sets mu-inariants to 0, these shoudl be constant!!
    classic_constant = all((b - bodies[0]) == 0 for b in bodies) # it checkt that boedies is a constant vector for sanity check, should always be true, if not something is wrong with the code...
    return {
        'P': P,
        'full_constant': full_constant,
        'classic_constant': classic_constant,
        'body_value': bodies[0],
        'values': vals[0] if full_constant else vals
    }

#This auxiliary function helps pass which growth recurssion (even, odd, half_odd) you want to check
def _entry(L, E, i, j, kind, n, ap):
    if kind == 'odd_half':
        return super_entry_half(L, E, i, j, n, ap) #ap is short for the anti_periodic boolean
    return super_entry(L, E, i, j, n, ap)[2 if kind == 'odd' else 1]


def check_recursion(L, E, F, n=None, kind='even',
                    irange=range(0, 3), jrange=range(0, 5), odd_antiperiodic=False):
    """Theorem-2.5 analogue at period P:
        do we have  entry(i, j+2P) = s * entry(i, j+P) - entry(i, j) ? 

    Returns the list of (i, j, residual) failures; empty means it holds on the window.
    """
    if n is None:
        n = len(L)
    P = superfrieze_period(n, odd_antiperiodic)
    fn = _growth_fn(kind)
    s = fn(L, E, 0, n, odd_antiperiodic) # I guess testing it for the odd rows is king of pointless since it seems that the "growth" values computed with this code are not constant. But this does it anyway :P
    proj = (lambda x: F(x)) #Defines de coercion function into F
    fails = []
    for i in irange:
        for j in jrange:
            lhs = proj(_entry(L, E, i, j + 2 * P, kind, n, odd_antiperiodic))
            rhs = proj(s) * proj(_entry(L, E, i, j + P, kind, n, odd_antiperiodic)) - proj(_entry(L, E, i, j, kind, n, odd_antiperiodic))
            d = lhs - rhs
            bad = (not _is_zero(d, F))
            if bad:
                fails.append((i, j, d))
    return fails


In [3]:
#Example of a annulus with 3 marked points in the outer boundary and 1 in the inner boundary. The triangulation has quiddity sequence associated to the outer boundary (3, 2, 2). 
F = superalgebra(4,4)
MM = MatrixSpace(F, 3, sparse=False)

#Definition of the basic holonomy matrices. Since the code does not support sqrt, insert the variable as you would in paper, the definition of the matrices accounts for this.
def E(param):
    param=F(param)
    return MM([[0,-param^2,0],[1/param^2,0,0],[0,0,1]])

def A(h, angle):
    h = F(h)
    angle = F(angle)
    return MM([[1,0,0], [h^2, 1, -h*angle], [h*angle, 0, 1]])
# E inverse
def EI(param):
    param = F(param)
    return MM([[0,param^2,0],[-1/param^2,0,0],[0,0,1]])
# A inverse
def AI(h, angle):
    h = F(h)
    angle = F(angle)
    return MM([[1,0,0], [-h^2, 1, h*angle], [-h*angle, 0, 1]])

R = MM([[-1,0,0],[0,-1,0],[0,0,1]])

#My Sage is buggy for some reason, so I have to redefine matrix multiplication for matrices over an exterior algebra
def mat_mult(A, B):
    n = A.nrows()
    result = [[sum(A[i,k] * B[k,j] for k in range(n)) for j in range(n)] for i in range(n)]
    return MM(result)

def mat_prod(*args):
    result = args[0]
    for m in args[1:]:
        result = mat_mult(result, m)
    return result

# Code for a fan of n triangles of around a marked point x of a skeletal triangulation
# Here T is the vector of mu-invariants associated to the triangles of the fan
# L are the even elements (opposite edge to x)/(product of adjacent edges to x)
# Note that when all lambda lenghts are set to 1, the entries of the following matrix are exactly the ones obtained in Proposition 3.4 of our submission
def holonomy_fan(n, L, T):
    H = E(1)
    for i in range(n) : 
        H = mat_mult(A(L[i], T[i]), H)
    H = mat_mult(E(1), H)
    return [H[0,1], H[0,2]]


### Example : 3-periodic super frieze coming from a (1, 3) annulus

In [4]:
#Compute the quiddity sequences associated to the outer boundary, we use the triangulation in figure 9, but for any generators. We number the arcs x1, x2, x3, x4, where x1 and x4 are the two arcs coming out of the marked point (1), x2 is the one adjacent to (2), and x3 is the one adjacent to (3).
# For the mu invariants, T1, T2 and T3 are the the "downwards" triangles adjacent to (1,2), (2,3) and (3,1). The variable T4 is associated to the upwards triangle. 
fan1 = holonomy_fan(2, [x1/x2, x3/x2], [T1,T2])
fan2 = holonomy_fan(2, [x2/x3, x4/x3], [T2,T3])
fan3 = holonomy_fan(3, [x3/x4, 1/(x1*x4), x2/x1], [T3,T4,T1])
L = [fan1[0], fan2[0], fan3[0]]     # even quiddity sequence  (period P = 3)
T = [fan1[1], fan2[1], fan3[1]]     # odd  quiddity sequence
P = len(L)

O1 = even_growth(L, T, 0, P)
O2 = even_growth(L, T, 1, P)
# The followin code prints the difference between an entry in the 3rd non even row and the entry above it, as well as the ifference for the following entry in the same row. It is not constant. 
slatex(O1, O2, O1 == O2)   #not constant

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
#Hoewver, when you make all odd variables equal t 0, it is contant and you recover the classical notion of the principal growth coefficient. 
slatex(body(O1, F) == body(O2, F), body(O1, F))

<IPython.core.display.Math object>

<IPython.core.display.Math object>